In [1]:
# If ipywidgets is missing, uncomment and run the following line once:
# !pip install ipywidgets

from IPython.display import display, HTML, clear_output
import ipywidgets as widgets
import datetime
import pandas as pd

# Try to detect Colab download support
try:
    from google.colab import files
    in_colab = True
except Exception:
    in_colab = False

# --- 50 questions (same wording as before) ---
questions = [
 "Do you feel energetic throughout the day?",
 "Do you sleep at least 7–8 hours daily?",
 "Do you feel satisfied with your daily routine?",
 "Do you often feel physically tired without reason?",
 "Do you find time for hobbies/relaxation?",
 "Do you feel motivated in the morning?",
 "Are you able to concentrate on tasks?",
 "Do you feel happy most days?",
 "Do you maintain a balanced diet?",
 "Do you exercise regularly?",

 "Do you feel overwhelmed by responsibilities?",
 "Do you find it hard to relax after work/study?",
 "Do you often feel restless or uneasy?",
 "Do you get irritated easily?",
 "Do you feel pressure from deadlines?",
 "Do you feel like you don’t have control over your life?",
 "Do you experience frequent headaches or muscle tension?",
 "Do you worry about things outside your control?",
 "Do you often feel mentally exhausted?",
 "Do you struggle to balance personal and professional life?",

 "Do you feel nervous in social situations?",
 "Do you worry excessively about small issues?",
 "Do you experience shortness of breath when anxious?",
 "Do you avoid situations out of fear?",
 "Do you feel panic or sudden fear without reason?",
 "Do you overthink conversations after they happen?",
 "Do you feel your heart racing without physical activity?",
 "Do you often anticipate the worst outcome?",
 "Do you sweat or shake in stressful situations?",
 "Do you have difficulty making decisions due to fear?",

 "Do you feel hopeless about the future?",
 "Do you lack interest in activities you used to enjoy?",
 "Do you feel worthless or guilty often?",
 "Do you experience sudden mood swings?",
 "Do you struggle to get out of bed in the morning?",
 "Do you have trouble sleeping (too little/too much)?",
 "Do you feel sad most of the time?",
 "Do you experience frequent crying spells?",
 "Do you avoid social interaction?",
 "Do you ever feel life is not worth living?",

 "Do you feel supported by friends/family?",
 "Do you have someone you can talk to when upset?",
 "Do you use unhealthy coping strategies (alcohol, smoking, overeating)?",
 "Do you engage in positive self-care (meditation, journaling)?",
 "Do you manage time effectively?",
 "Do you have a good work-life balance?",
 "Do you feel financially stressed?",
 "Do you rely on spiritual/religious practices for peace?",
 "Do you feel isolated or lonely?",
 "Do you think professional help would benefit you?"
]

# --- Build UI widgets ---
options = [("1 - Never", 1), ("2 - Rarely", 2), ("3 - Sometimes", 3), ("4 - Often", 4), ("5 - Always", 5)]
dropdowns = []
for i, q in enumerate(questions):
    dd = widgets.Dropdown(options=options, value=3, description=f"Q{i+1}", layout=widgets.Layout(width='420px'), tooltip=q)
    dropdowns.append(dd)

# arrange two columns per row for compactness
rows = []
for i in range(0, len(dropdowns), 2):
    left = dropdowns[i]
    right = dropdowns[i+1] if i+1 < len(dropdowns) else widgets.Label('')
    rows.append(widgets.HBox([left, right]))

grid = widgets.VBox(rows, layout=widgets.Layout(max_height='600px', overflow='auto'))

submit_btn = widgets.Button(description="Submit", button_style='primary')
save_btn = widgets.Button(description="Save CSV & Download", disabled=True)
out = widgets.Output()

# helper functions
positive_items = {0,1,2,4,5,6,7,8,9,40,41,43,44,45,47}  # 0-based indices for positive-worded Qs
def reverse_score(x): return 6 - x

def category_band(avg):
    if avg <= 2.0: return "Low"
    elif avg <= 3.0: return "Moderate"
    elif avg <= 4.0: return "High"
    else: return "Very High"

categories = {
    "General Well-being": list(range(0,10)),
    "Stress Level": list(range(10,20)),
    "Anxiety Symptoms": list(range(20,30)),
    "Depression Symptoms": list(range(30,40)),
    "Coping & Support": list(range(40,50))
}

last_result = {}  # will hold results for saving

def on_submit(btn):
    with out:
        clear_output()
        responses = [d.value for d in dropdowns]
        # scoring (reverse positive items)
        scored = [reverse_score(val) if i in positive_items else val for i,val in enumerate(responses)]
        total_score = sum(scored)
        if total_score <= 100:
            overall = "Healthy Mental State"
        elif total_score <= 150:
            overall = "Mild Concerns"
        elif total_score <= 200:
            overall = "Moderate Concerns"
        else:
            overall = "Severe Concerns"

        # build report html
        html = f"<h2>Mental Health Assessment — Report</h2>"
        html += f"<b>Date:</b> {datetime.datetime.now().isoformat(sep=' ', timespec='seconds')}<br>"
        html += f"<b>Total Score:</b> {total_score} (range 50–250)<br>"
        html += f"<b>Overall Status:</b> {overall}<br><hr>"

        html += "<h3>Category breakdown</h3>"
        cat_details = {}
        for cat, idxs in categories.items():
            s = sum(scored[i] for i in idxs)
            avg = s / len(idxs)
            band = category_band(avg)
            html += f"<b>{cat}:</b> {s} points (avg {avg:.2f}) → {band}<br>"
            cat_details[cat] = {"total": s, "avg": avg, "band": band}

        html += "<hr><h3>Recommendations</h3>"
        if overall == "Healthy Mental State":
            html += "<ul><li>Maintain current healthy habits and keep monitoring your well-being.</li></ul>"
        elif overall == "Mild Concerns":
            html += "<ul><li>Practice sleep hygiene, stress-management (breathing, short walks), and regular self-care.</li></ul>"
        elif overall == "Moderate Concerns":
            html += "<ul><li>Talk to trusted people, try structured self-help, and consider counseling if it persists.</li></ul>"
        else:
            html += "<ul><li>Strongly advised to consult a mental health professional. If you're in immediate danger, contact emergency services.</li></ul>"

        # urgent safety check (Q40 is index 39)
        if responses[39] >= 4:
            html += "<hr><h3 style='color:red'>*** URGENT SAFETY NOTICE ***</h3>"
            html += "<p style='color:red'>You indicated frequent thoughts that life is not worth living. Please contact emergency services, a crisis hotline, or a mental health professional immediately.</p>"

        display(HTML(html))

        # enable save button and store last_result
        save_btn.disabled = False
        last_result.clear()
        last_result.update({
            "timestamp": datetime.datetime.now().isoformat(sep=' ', timespec='seconds'),
            "responses": responses,
            "scored": scored,
            "total_score": total_score,
            "overall": overall,
            "categories": cat_details
        })

def on_save(btn):
    # Build a single-row DataFrame for easy CSV export
    if not last_result:
        with out:
            print("No results to save. Click Submit first.")
        return

    row = {
        "timestamp": last_result["timestamp"],
        "total_score": last_result["total_score"],
        "overall": last_result["overall"]
    }
    # add category totals
    for cat, v in last_result["categories"].items():
        row[f"{cat}_total"] = v["total"]
        row[f"{cat}_avg"] = v["avg"]
        row[f"{cat}_band"] = v["band"]

    # add raw and scored Qs
    for i, val in enumerate(last_result["responses"], start=1):
        row[f"Q{i}_raw"] = val
    for i, val in enumerate(last_result["scored"], start=1):
        row[f"Q{i}_scored"] = val

    df = pd.DataFrame([row])
    fname = f"mha_results_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
    df.to_csv(fname, index=False)

    with out:
        print(f"Saved CSV to: {fname}")
        if in_colab:
            try:
                files.download(fname)
            except Exception as e:
                print("Download failed automatically; you can find the file in the Colab file browser (/content).", e)

submit_btn.on_click(on_submit)
save_btn.on_click(on_save)

display(HTML("<h1>Mental Health Assessment (Interactive)</h1><p>Answer each dropdown then press <b>Submit</b>.</p>"))
display(grid)
display(widgets.HBox([submit_btn, save_btn]))
display(out)


Output()